# RSNA Knee — 제출용 추론 Notebook

이 Notebook을 **Internet Off** 상태로 Save & Run All 한 뒤, 성공한 버전의 `submission.csv`를 제출하세요.

- 학습 Notebook v4의 모델을 사용하며 재학습하지 않습니다.
- 대회 데이터와 **RSNA Knee Offline Assets v1** 출력이 연결돼 있어야 합니다.
- 테스트 판독문이나 학습 데이터의 정답을 사용하지 않습니다.
- 공개 예제의 3개 검사에 한정하지 않고, 채점 시 주어지는 `test.csv` 전체를 처리합니다.
- 모델은 43개 검사로 학습한 초기 기준 모델입니다. 제출 가능 여부와 예측 성능은 별개입니다.

데이터·제출 조건: https://www.kaggle.com/competitions/rsna-knee-abnormality-detection/overview/evaluation


In [ ]:
from pathlib import Path
import hashlib
import json
import subprocess
import sys
import time

started = time.monotonic()
input_root = Path("/kaggle/input")
asset_roots = [input_root, input_root / "notebooks" / "choiyoungwhan",
               input_root / "kernels" / "choiyoungwhan"]
manifests = [p for root in asset_roots for p in root.glob("*/assets.json")]
assert len(manifests) == 1, "Attach the completed RSNA Knee Offline Assets notebook output."
ASSETS = manifests[0].parent
manifest = json.loads(manifests[0].read_text(encoding="utf-8"))
assert list(sys.version_info[:2]) == manifest["python"], "Use the same Docker image as Offline Assets."
checkpoint_path = ASSETS / "baseline.pt"
assert hashlib.sha256(checkpoint_path.read_bytes()).hexdigest() == manifest["checkpoint_sha256"]
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "--no-index", "--no-deps",
    "--find-links", str(ASSETS / "wheels"), "pydicom==3.0.2", "python-gdcm==3.2.6",
])

import numpy as np
import pandas as pd
import pydicom
import torch
from torch import nn
from torchvision.models import ResNet18_Weights, resnet18
from tqdm.auto import tqdm

candidates = [input_root / "rsna-knee-abnormality-detection",
              input_root / "competitions" / "rsna-knee-abnormality-detection"]
DATA = next((p for p in candidates if (p / "test.csv").exists()), None)
assert DATA is not None, "Attach the competition data."
OUT = Path("/kaggle/working")
OUT.mkdir(exist_ok=True)

checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
labels = checkpoint["labels"]
PLANES = tuple(checkpoint["planes"])
assert len(labels) == 12 and PLANES == ("Sagittal", "Coronal", "Axial")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# weights=None prevents a network download; trained weights come from the attached checkpoint.
encoder = resnet18(weights=None)
encoder.fc = nn.Identity()
encoder.load_state_dict(checkpoint["encoder"], strict=True)
encoder = encoder.to(device).eval()
encoder.requires_grad_(False)
transform = ResNet18_Weights.IMAGENET1K_V1.transforms()
head = nn.Linear(1536, len(labels))
head.load_state_dict(checkpoint["head"], strict=True)
head = head.to(device).eval()
mean, std = checkpoint["mean"].to(device), checkpoint["std"].to(device)
assert mean.shape == std.shape == (1536,)
assert torch.isfinite(mean).all() and torch.isfinite(std).all() and (std > 0).all()
print("Device:", torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU")


In [ ]:
def ordered_slices(folder):
    paths = sorted(folder.glob("*.dcm"))
    if not paths:
        raise FileNotFoundError(f"No DICOM slices in {folder}")
    headers = [pydicom.dcmread(path, stop_before_pixels=True,
                              specific_tags=["ImagePositionPatient", "ImageOrientationPatient", "InstanceNumber"])
               for path in paths]
    if all(hasattr(h, "ImagePositionPatient") and hasattr(h, "ImageOrientationPatient") for h in headers):
        orientation = np.asarray(headers[0].ImageOrientationPatient, dtype=float)
        normal = np.cross(orientation[:3], orientation[3:])
        assert np.linalg.norm(normal) > 0.9, "Invalid DICOM orientation."
        positions = [float(np.dot(np.asarray(h.ImagePositionPatient, dtype=float), normal)) for h in headers]
    elif all(hasattr(h, "InstanceNumber") for h in headers):
        positions = [float(h.InstanceNumber) for h in headers]
    else:
        raise ValueError(f"Missing slice position and InstanceNumber: {folder}")
    return [paths[i] for i in np.argsort(positions, kind="stable")]


def image_tensor(path):
    ds = pydicom.dcmread(path)
    pixels = ds.pixel_array.astype(np.float32)
    assert pixels.ndim == 2 and np.isfinite(pixels).all(), f"Invalid MRI slice: {path}"
    pixels = pixels * float(getattr(ds, "RescaleSlope", 1)) + float(getattr(ds, "RescaleIntercept", 0))
    if getattr(ds, "PhotometricInterpretation", "") == "MONOCHROME1":
        pixels = -pixels
    low, high = np.percentile(pixels, [1, 99])
    if high <= low:
        low, high = float(pixels.min()), float(pixels.max())
    if high <= low:
        raise ValueError(f"Constant-valued MRI slice: {path}")
    pixels = np.clip((pixels - low) / (high - low), 0, 1)
    tensor = torch.from_numpy(pixels).float().unsqueeze(0).repeat(3, 1, 1)
    return transform(tensor)


@torch.inference_mode()
def study_features(study_uid, series_table, image_root):
    rows = series_table.loc[series_table.StudyInstanceUID == study_uid]
    assert len(rows), f"No series metadata for {study_uid}"
    features = []
    used_planes = 0
    for plane in PLANES:
        choices = rows.loc[rows.Anatomical_Plane == plane].sort_values(
            ["Fluid_Sensitive", "Fat_Suppression", "SeriesInstanceUID"], ascending=[False, False, True])
        if choices.empty:
            features.append(torch.zeros(512))
            continue
        row = choices.iloc[0]
        paths = ordered_slices(image_root / study_uid / row.SeriesInstanceUID)
        # shortcut: three slices from one series per plane; expand coverage before competitive training.
        indices = np.unique(np.rint(np.linspace(0.25, 0.75, 3) * (len(paths) - 1)).astype(int))
        batch = torch.stack([image_tensor(paths[i]) for i in indices]).to(device)
        features.append(encoder(batch).mean(dim=0).cpu())
        used_planes += 1
    assert used_planes, f"No recognized anatomical planes for {study_uid}"
    result = torch.cat(features)
    assert result.shape == (1536,) and torch.isfinite(result).all()
    return result


In [ ]:
test = pd.read_csv(DATA / "test.csv")
test_series = pd.read_csv(DATA / "test_series.csv")
sample = pd.read_csv(DATA / "sample_submission.csv")
assert test.StudyInstanceUID.notna().all() and test.StudyInstanceUID.is_unique
assert len(test) > 0 and test_series.SeriesInstanceUID.is_unique
assert set(test.StudyInstanceUID) <= set(test_series.StudyInstanceUID)
assert sample.columns.tolist() == ["StudyInstanceUID", *labels]

# Read every current test ID; Kaggle replaces the example test data during scoring.
probabilities = []
with torch.inference_mode():
    for uid in tqdm(test.StudyInstanceUID, desc="Predicting test MRI studies"):
        features = study_features(uid, test_series, DATA / "test_series").to(device)
        prediction = head(((features - mean) / std).unsqueeze(0)).sigmoid().cpu().numpy()[0]
        probabilities.append(prediction)
probabilities = np.stack(probabilities)
assert probabilities.shape == (len(test), len(labels))
assert np.isfinite(probabilities).all() and ((probabilities >= 0) & (probabilities <= 1)).all()
submission = pd.DataFrame(probabilities, columns=labels)
submission.insert(0, "StudyInstanceUID", test.StudyInstanceUID.to_numpy())
assert submission.columns.tolist() == sample.columns.tolist()
assert submission.StudyInstanceUID.tolist() == test.StudyInstanceUID.tolist()
target = OUT / "submission.csv"
temporary = OUT / "submission.csv.part"
submission.to_csv(temporary, index=False)
temporary.replace(target)
saved = pd.read_csv(target)
assert saved.shape == (len(test), 13) and not saved.isna().any().any()
assert saved.StudyInstanceUID.tolist() == test.StudyInstanceUID.tolist()
summary = {"studies": len(test), "labels": len(labels),
           "seconds": round(time.monotonic() - started, 2),
           "checkpoint_sha256": manifest["checkpoint_sha256"],
           "output": target.name}
(OUT / "inference_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(json.dumps(summary, indent=2))
print(submission.head().to_string(index=False))
